# Non-reversible anchored Langevin with a block state-dependent skew-symmetric matrix

Synthetic Bayesian logistic regression under two constraint sets, comparing four
samplers that share a single implementation and differ only in two parameters.

| Method | $\rho$ | $\alpha$ |
|---|---:|---:|
| Projected SGLD | $0$ | $0$ |
| Non-reversible SGLD | $0$ | $1$ |
| Reversible anchored Langevin | $\log 2$ | $0$ |
| Non-reversible anchored Langevin | $\log 2$ | $1$ |

The two anchored rows isolate the effect of non-reversibility **at a fixed anchor**;
the two $\alpha=0$ rows isolate the effect of the anchor at fixed reversibility.

**The deliverable is the measurement, not a favourable result.** Nothing here assumes
that non-reversibility or anchoring must help. The observed outcome at the paper's
candidate step size is reported as found, together with a diagnosis and a step-size
refinement that changes the conclusion.

Sections follow the requested order: configuration and seeds, data, posterior and
mini-batch gradient, constraints and projections, anchor, block matrices,
implementation checks, sampler, repeated experiments, accuracy plots, step-size
sensitivity, interpretation and limitations.

## 1. Configuration and seeds

Main settings: $d=9$, $n_{\text{total}}=2000$, stratified 80/20 split, mini-batch
$m=50$, 1000 iterations, candidate step size $h=10^{-4}$, $R=100$ replicates,
block strengths $s=(10,10,10)$, $\rho=\log 2$ for the anchored methods.

Seeds are fixed: `data_seed = 2026`, `split_seed = 2027`, `sampler_seed = 3000`.
The data set and split are frozen across every method, replicate and geometry, so
the replicate spread measures **sampler randomness only**.

Set the environment variable `NRAL_QUICK=1` before launching Jupyter to run quick
mode (5 repeats, 300 iterations). Quick-mode artefacts are written with a
`_quick` suffix and every printed table is tagged, so they can never be confused
with the full experiment.

In [ ]:
import os, sys, time, json, platform, warnings
import numpy as np, pandas as pd, scipy, sklearn, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
import inspect

sys.path.insert(0, os.path.abspath("."))
import anchored_sgld as nral

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight"})

QUICK = os.environ.get("NRAL_QUICK", "0") == "1"
MODE = "QUICK MODE" if QUICK else "FULL EXPERIMENT"
SUFFIX = "_quick" if QUICK else ""
OUTPUT_DIR = "results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

cfg = nral.ExperimentConfig(
    n_repeats=5 if QUICK else 100,
    n_iterations=300 if QUICK else 1000,
    checkpoint_every=10,
    sensitivity_repeats=3 if QUICK else 20,
)
cfg_d3 = nral.ExperimentConfig(
    d=3, block_scales=(10.0,),
    n_repeats=cfg.n_repeats, n_iterations=cfg.n_iterations,
    sensitivity_repeats=cfg.sensitivity_repeats,
)

def show(obj):
    display(Markdown("```python\n" + inspect.getsource(obj) + "\n```"))

print(f"=== {MODE} ===")
print(f"d = {cfg.d}, n_total = {cfg.n_total}, n_train = {cfg.n_train}, n_test = {cfg.n_test}")
print(f"m = {cfg.batch_size}, iterations = {cfg.n_iterations}, h = {cfg.step_size}, R = {cfg.n_repeats}")
print(f"block strengths s = {tuple(float(x) for x in cfg.scales)}  (n_blocks = {cfg.n_blocks})")
print(f"rho (anchored) = log 2 = {nral.RHO_ANCHORED:.6f};  rho (unanchored) = 0")
print(f"quartic: epsilon = {cfg.epsilon}, Lambda = {cfg.Lambda}")
print(f"target accuracy for time-to-target reporting = {cfg.target_accuracy}")
print(f"\nseeds: data={cfg.data_seed}, split={cfg.split_seed}, sampler={cfg.sampler_seed}")
VERSIONS = {"python": platform.python_version(), "numpy": np.__version__,
            "scipy": scipy.__version__, "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__, "matplotlib": matplotlib.__version__}
print("versions:", VERSIONS)

## 2. Synthetic data generation

$$X_j\sim N(0,2I_d),\qquad u_j\sim\text{Uniform}(0,1),\qquad
y_j=\mathbf 1\{u_j\le\sigma(X_j^\top\beta_{\text{true}})\}.$$

The coordinate standard deviation is $\sqrt2$. No intercept, no feature
standardisation. The constraints act on the **coefficients** $\beta$, never on the
observed feature vectors $X_j$.

`beta_true` is the fixed generating vector, used once to draw the labels and never
seen by the sampler. `beta` is the unknown coefficient the sampler explores.

In [ ]:
dataset = nral.make_dataset(cfg)
ball = nral.make_geometry("ball", cfg)
quartic = nral.make_geometry("quartic", cfg)
beta_true = dataset.beta_true

print(f"X_train {dataset.X_train.shape}, X_test {dataset.X_test.shape}")
print(f"train positives {dataset.y_train.mean():.4f}, test positives {dataset.y_test.mean():.4f}"
      "   (stratified, so these match)")
print(f"empirical coordinate sd of X: {dataset.X_train.std(axis=0).mean():.4f}  (target sqrt(2) = {np.sqrt(2):.4f})")
print(f"\nbeta_true = {np.round(beta_true, 3).tolist()}")

feas = pd.DataFrame([
    {"constraint": "ball  ||beta||^2 <= 1", "value": float(ball.constraint_value(beta_true)),
     "threshold": ball.threshold, "feasible": bool(ball.feasible(beta_true))},
    {"constraint": "quartic  g(beta) <= Lambda", "value": float(quartic.constraint_value(beta_true)),
     "threshold": quartic.threshold, "feasible": bool(quartic.feasible(beta_true))},
])
display(feas)
print(f"g_min = d*eps^4 = {quartic.g_min:.4f}   (0.0144 for d=9);  D = Lambda - g_min = {quartic.D:.4f}")
assert ball.feasible(beta_true) and quartic.feasible(beta_true)

# Accuracy ceiling, so the 'stated accuracy' target below is interpretable.
from scipy.special import expit
p_test = expit(dataset.X_test @ beta_true)
print(f"\naccuracy of beta_true itself : train {nral.accuracy(beta_true, dataset.X_train, dataset.y_train):.4f}"
      f"  test {nral.accuracy(beta_true, dataset.X_test, dataset.y_test):.4f}")
print(f"Bayes ceiling E[max(p,1-p)]  : {np.mean(np.maximum(p_test, 1 - p_test)):.4f}")
print("The problem is intrinsically noisy: ~0.66 test accuracy is the ceiling, not an underfit.")

## 3. Posterior and mini-batch gradient

$$\pi_K(\beta)\propto e^{-U(\beta)}\mathbf 1_K(\beta),\qquad
U(\beta)=\sum_{j\in\text{train}}\bigl[\log(1+e^{X_j^\top\beta})-y_jX_j^\top\beta\bigr].$$

A **sum**, not a mean. `np.logaddexp(0, z)` is the stable softplus and the
label-dependent term is retained. The prior is uniform on $K$, which contributes
nothing to $\nabla U$ inside $K$ and is enforced entirely by the projection.

For a uniform mini-batch $B_k$ of size $m$, drawn without replacement within the
iteration,
$$\widehat G_k=\frac{n_{\text{train}}}{m}X_{B_k}^\top\bigl[\sigma(X_{B_k}\beta_k)-y_{B_k}\bigr].$$
The $n_{\text{train}}/m$ factor is what makes this unbiased for the *summed*
gradient; §7 verifies it by enumerating every batch of a toy data set.

In [ ]:
show(nral.potential_U); show(nral.minibatch_gradient)
print(f"U(beta_true)          = {nral.potential_U(beta_true, dataset.X_train, dataset.y_train):,.3f}")
print(f"||grad U(beta_true)|| = {np.linalg.norm(nral.full_gradient(beta_true, dataset.X_train, dataset.y_train)):,.3f}")

## 4. Constraints and projections

$$K_{\text{ball}}=\{\beta:\|\beta\|_2^2\le1\},\qquad
K_{\text{quartic}}=\Bigl\{\beta:g(\beta)=\sum_{i=1}^d(\beta_i^2+\varepsilon^2)^2\le\Lambda\Bigr\}$$
with $\varepsilon=0.2$, $\Lambda=1$.

**Ball.** $\Pi_K(z)=z$ if $\|z\|\le1$, else $z/\|z\|$.

**Quartic.** Return $z$ if feasible. Otherwise solve the Euclidean projection
equations
$$b_i+4\mu b_i(b_i^2+\varepsilon^2)=z_i,\qquad g(b)=\Lambda,\qquad\mu>0.$$
For fixed $\mu$ the coordinate map $b\mapsto b+4\mu b(b^2+\varepsilon^2)$ is odd
and strictly increasing (derivative $1+12\mu b^2+4\mu\varepsilon^2>0$), so each
coordinate has a unique root, obtained in closed form from the depressed cubic
via the numerically stable hyperbolic branch. The outer map $\mu\mapsto g(b(\mu))$
is decreasing, so $\mu$ is found by a bracketed bisection.

**Radial scaling is not used.** It is feasible but is not the Euclidean projection
unless the set is a Euclidean ball; §7 shows it is strictly worse on the quartic
set in every projected case.

In [ ]:
show(nral.QuarticGeometry.project)

rng_demo = np.random.default_rng(5)
z = rng_demo.normal(scale=1.6, size=cfg.d)
for geometry in (ball, quartic):
    out = geometry.project(z)
    print(f"{geometry.name:<12} g(z) = {float(geometry.constraint_value(z)):8.4f} -> "
          f"g(proj) = {float(geometry.constraint_value(out.beta)):.10f} "
          f"(threshold {geometry.threshold})  KKT residual {out.max_kkt_residual:.2e}")

## 5. Anchor and anchoring coefficient

$$U_0(\beta)=U(\beta)+\rho H_K(\beta),\qquad
a(\beta)=e^{U(\beta)-U_0(\beta)}=e^{-\rho H_K(\beta)}.$$

* Ball: $H_K(\beta)=\|\beta\|^2$, $\nabla H_K=2\beta$.
* Quartic: $H_K(\beta)=\dfrac{g(\beta)-d\varepsilon^4}{D}$, $D=\Lambda-d\varepsilon^4$,
  $\partial_ig=4\beta_i(\beta_i^2+\varepsilon^2)$, $\nabla H_K=\nabla g/D$.

Both give $H_K\in[0,1]$ on $K$, so with $\rho=\log2$ we get $\tfrac12\le a\le1$.

$a$ is computed **exactly from the geometry**. A noisy estimate of the likelihood
difference is never exponentiated — that would be both biased (by Jensen) and
explosive for a sum over 1600 rows.

The stochastic anchor gradient is $v_k=\widehat G_k+\rho\nabla H_K(\beta_k)$: the
anchor derivative is exact and added **once**, and is *not* multiplied by
$n_{\text{train}}/m$, because it is not estimated from the batch.

**This is a proposed non-trivial anchor for an already smooth target, not an extra
Bayesian penalty.** The target $\pi_K$ is unchanged: $a$ enters the drift and the
diffusion coefficient in the one combination that leaves $e^{-U}$ invariant.

In [ ]:
print(nral.invariant_measure_note())

## 6. Block state-dependent skew-symmetric matrices

$$[w]_\times=\begin{pmatrix}0&-w_3&w_2\\ w_3&0&-w_1\\ -w_2&w_1&0\end{pmatrix},
\qquad [w]_\times v=w\times v.$$

Coordinate triples $I_1=(1,2,3)$, $I_2=(4,5,6)$, $I_3=(7,8,9)$.

* Ball: $J_s(\beta)=\operatorname{blockdiag}\bigl([s_1\beta_{I_1}]_\times,\;
  [s_2\beta_{I_2}]_\times,\;[s_3\beta_{I_3}]_\times\bigr)$.
* Quartic: $J_g(\beta)_{I_\ell,I_\ell}=[-s_\ell\nabla_{I_\ell}g(\beta)]_\times$.

**Why the quartic needs $\nabla g$.** Tangency $Jn=0$ holds because
$[w]_\times w=w\times w=0$, so the block vector must be parallel to the normal.
The ball's normal is parallel to $\beta$; the quartic's is parallel to $\nabla g$,
whose coordinates are $\beta_i$ reweighted by $4(\beta_i^2+\varepsilon^2)$. These
directions differ unless all $|\beta_i|$ are equal, so the unmodified ball matrix
generally fails $Jn=0$ on the quartic boundary — demonstrated numerically below.

Sampling uses matrix-free block cross products; an explicit-matrix helper exists
only for the checks. Each block strength multiplies its block vector exactly once.

In [ ]:
show(nral.apply_J)
b_demo = ball.sample_uniform(np.random.default_rng(3), 1)[0]
J_demo = nral.build_J(b_demo, ball, cfg.scales)
print("J(beta) on the ball (block diagonal, skew-symmetric):")
print(np.array2string(J_demo, precision=3, suppress_small=True))

fail = nral.check_ball_J_fails_on_quartic(cfg)
print(f"\nOn the quartic boundary, max |J n|:")
print(f"  correct quartic J (uses -s grad g) : {fail['quartic_J_tangency']:.3e}  -> tangential")
print(f"  unmodified ball J (uses s beta)    : {fail['ball_J_on_quartic_boundary']:.3e}  -> NOT tangential")

## 7. Mathematical implementation checks

Run before any production sampling. Every check is an assertion, so the notebook
fails loudly rather than producing plausible-looking wrong numbers.

In [ ]:
checks = {}
checks["gradient vs finite differences"] = nral.check_gradient_finite_difference(dataset)
checks["mini-batch scaling (all batches enumerated)"] = nral.check_minibatch_scaling_exhaustive()
for geometry in (ball, quartic):
    tag = geometry.name
    checks[f"J identities [{tag}]"] = nral.check_matrix_identities(geometry, cfg.scales)
    checks[f"anchor bounds [{tag}]"] = nral.check_anchor_bounds(geometry)
    checks[f"projection [{tag}]"] = nral.check_projection(geometry)
    checks[f"J grad_U0 nonzero [{tag}]"] = nral.check_J_gradU0_nonzero(dataset, geometry, cfg.scales)
checks["projection solvers agree [quartic]"] = nral.check_projection_solvers_agree(quartic)
checks["ball J on quartic boundary"] = nral.check_ball_J_fails_on_quartic(cfg)

for name, values in checks.items():
    formatted = ", ".join(f"{k} = {v:.3e}" if abs(v) < 1e-3 or abs(v) > 1e4 else f"{k} = {v:.6g}"
                          for k, v in values.items())
    print(f"{name}\n    {formatted}")

# --- assertions ---
assert checks["gradient vs finite differences"]["max_relative_error"] < 1e-6
mb = checks["mini-batch scaling (all batches enumerated)"]
assert mb["relative_error_scaled"] < 1e-12, "n_train/m scaling is wrong"
assert mb["max_abs_error_if_unscaled"] > 1e-3, "the unscaled contrast should be far off"
for geometry in (ball, quartic):
    tag = geometry.name
    ident = checks[f"J identities [{tag}]"]
    assert ident["max_skew_error"] < 1e-12          # J^T = -J
    assert ident["max_divergence"] < 1e-8           # div J = 0
    assert ident["max_tangency_on_boundary"] < 1e-10  # J n = 0 on the boundary
    assert ident["max_J_gradH"] < 1e-10             # J grad_H = 0
    assert ident["max_matrix_free_vs_explicit"] < 1e-10
    assert checks[f"anchor bounds [{tag}]"]["bounds_hold"] == 1.0    # 1/2 <= a <= 1
    proj = checks[f"projection [{tag}]"]
    assert proj["max_kkt_residual"] < 1e-8 and proj["max_feasibility_excess"] < 1e-8
    # The non-reversible term must actually do something.
    assert checks[f"J grad_U0 nonzero [{tag}]"]["min_norm_J_gradU0"] > 0
assert checks["projection solvers agree [quartic]"]["max_abs_difference"] < 1e-10
assert checks["projection [quartic set]"]["n_radial_strictly_worse"] > 0, \
    "radial scaling should be strictly worse than the Euclidean projection here"
assert checks["ball J on quartic boundary"]["ball_J_on_quartic_boundary"] > 1e-3
print("\nAll implementation checks passed.")

## 8. The common sampler

$$\boxed{\;\beta_{k+1}=\Pi_K\!\left[\beta_k-ha_k\bigl(v_k+\alpha J(\beta_k)v_k\bigr)
+\sqrt{2ha_k}\,\xi_k\right]\;}$$

with $v_k=\widehat G_k+\rho\nabla H_K(\beta_k)$, $a_k=e^{-\rho H_K(\beta_k)}$ and
$\xi_k\sim N(0,I_d)$ drawn from a stream independent of the mini-batch stream.

No $\nabla a$ correction is added. $J$ is never rescaled beyond the constant block
strengths. All four methods are the same code path with different $(\rho,\alpha)$.

Within a replicate and geometry, all four methods receive identical starting
coefficients, identical mini-batch index sequences and identical Gaussian
increments; replicates use independent sub-streams.

**Initialisation** is uniform on the domain, and is a separate choice from the
Bayesian prior: ball, $ZV^{1/d}/\|Z\|$; quartic, rejection sampling from
$[-b,b]^d$ with $b=\sqrt{\sqrt{\Lambda-(d-1)\varepsilon^4}-\varepsilon^2}$.

In [ ]:
show(nral.run_sampler)
print(f"quartic rejection-sampling box half-width b = {quartic.box_half_width:.6f}")
_ = quartic.sample_uniform(np.random.default_rng(0), 2000)
print(f"quartic uniform-initialisation acceptance rate = {quartic.last_acceptance_rate:.4f}")

## 9. Repeated experiments

$R$ independent sampler replicates on the frozen data set and split.

In [ ]:
results = {}
for geometry in (ball, quartic):
    print(f"[{MODE}] geometry: {geometry.name}")
    results[geometry.name] = nral.run_all_methods(dataset, geometry, cfg)
    print()

summaries = {}
for label, runs in results.items():
    summaries[label] = nral.summarise(runs, cfg.target_accuracy)
    print(f"=== {label}  ({MODE}) ===")
    display(summaries[label][[
        "method", "rho", "alpha", "final_train_acc", "final_test_acc", "final_test_sd",
        "frac_reaching_target", "median_iters_to_target", "median_secs_to_target",
        "projection_rate", "drift_ratio_alphaJv_over_v", "mean_step_displacement",
        "n_nonfinite", "runtime_s"]])

### 9a. Why the non-reversible methods behave as they do

The continuous-time invariance argument assumes the **exact** gradient. With a
stochastic gradient, $J$ multiplies the gradient *error* as well as the gradient,
so the added term injects variance that the $\sqrt{2ha_k}$ term does not
compensate. The size of that effect is measured directly below.

In [ ]:
noise_rows = []
for geometry in (ball, quartic):
    report = nral.gradient_noise_report(dataset, geometry, cfg)
    report["geometry"] = geometry.name
    noise_rows.append(report)
noise_table = pd.DataFrame(noise_rows).set_index("geometry")
display(noise_table.T)
print("Read the last three rows: with h = 1e-4 the per-step displacement caused by")
print("J-amplified mini-batch noise is far larger than the injected Langevin noise,")
print("so at s = 10 the added term dominates the update rather than perturbing it.")

### 9b. Full-gradient control

Replacing $\widehat G_k$ by the exact $\nabla U$ removes the mini-batch noise while
changing nothing else. If the deficit is caused by noise amplification it should
shrink here; whatever survives is a property of the deterministic $\alpha Jv$ step.

In [ ]:
control = {}
for geometry in (ball, quartic):
    control[geometry.name] = nral.run_all_methods(
        dataset, geometry, cfg,
        n_iterations=min(cfg.n_iterations, 300 if QUICK else 1000),
        n_repeats=min(cfg.n_repeats, 20), use_full_gradient=True, verbose=False)
rows = []
for label, runs in control.items():
    frame = nral.summarise(runs, cfg.target_accuracy)
    frame.insert(0, "geometry", label)
    rows.append(frame)
control_table = pd.concat(rows, ignore_index=True)
display(control_table[["geometry", "method", "final_test_acc", "final_test_sd",
                       "projection_rate", "drift_ratio_alphaJv_over_v",
                       "mean_step_displacement"]])

### 9c. Block-strength sweep

$s$ is configurable independently. Only the product $\alpha s$ matters for the size
of the added term, so sweeping $s$ at $\alpha=1$ traces out the mechanism.

In [ ]:
sweep_rows = []
sweep_values = (0.0, 0.5, 1.0, 2.0, 5.0, 10.0) if not QUICK else (0.0, 1.0, 10.0)
for s_value in sweep_values:
    cfg_s = nral.ExperimentConfig(
        n_repeats=min(cfg.n_repeats, 20), n_iterations=cfg.n_iterations,
        block_scales=(s_value,) * cfg.n_blocks)
    for geometry in (ball, quartic):
        runs = nral.run_all_methods(dataset, geometry, cfg_s,
                                    n_repeats=min(cfg.n_repeats, 20), verbose=False)
        run = runs["Non-reversible anchored Langevin"]
        mean, sd = run.mean_std("test_accuracy")
        sweep_rows.append({"geometry": geometry.name, "s": s_value,
                           "final_test_acc": mean[-1], "final_test_sd": sd[-1],
                           "projection_rate": run.projection_rate,
                           "drift_ratio": run.drift_ratio,
                           "step_displacement": run.step_displacement})
sweep_table = pd.DataFrame(sweep_rows)
display(sweep_table.pivot(index="s", columns="geometry",
                          values=["final_test_acc", "projection_rate", "drift_ratio"]))
print("s = 0 reduces the non-reversible method to its reversible counterpart exactly.")

## 10. Accuracy plots

Single-iterate accuracy: at checkpoint $k$ each replicate predicts with **its own
current** $\beta_k$, $\hat y_{j,k}=\mathbf 1\{\sigma(X_j^\top\beta_k)\ge0.5\}$.
Coefficients are never averaged across replicates first, predictions are never
replaced by running averages, and no smoothing is applied.

Lines are the across-replicate mean; bands are mean $\pm$ one sample standard
deviation (`ddof=1`), clipped to $[0,1]$ **for display only**. These bands are
**repeat-run variability** — not confidence intervals and not posterior credible
intervals.

In [ ]:
figure_paths = []
for geometry, tag in ((ball, "fig1_ball"), (quartic, "fig2_quartic")):
    runs = results[geometry.name]
    figure_paths += nral.plot_accuracy_figure(
        runs, cfg, geometry.name, OUTPUT_DIR, f"{tag}_d{cfg.d}{SUFFIX}",
        title_extra="" if not QUICK else "  [QUICK MODE]")
    # Supplementary companion on an auto-scaled axis. Same data, same code path;
    # the [0, 1] figure above is the required one.
    lows, highs = [], []
    for run in runs.values():
        for which in ("train_accuracy", "test_accuracy"):
            mean, sd = run.mean_std(which)
            lows.append((mean - sd).min()); highs.append((mean + sd).max())
    pad = 0.02
    figure_paths += nral.plot_accuracy_figure(
        runs, cfg, geometry.name, OUTPUT_DIR, f"{tag}_d{cfg.d}{SUFFIX}",
        ylim=(max(0.0, min(lows) - pad), min(1.0, max(highs) + pad)),
        title_extra="  (supplementary: auto-scaled axis)" + (" [QUICK MODE]" if QUICK else ""))

for path in figure_paths:
    print("wrote", path)
for geometry, tag in ((ball, "fig1_ball"), (quartic, "fig2_quartic")):
    display(Markdown(f"**{geometry.name}**"))
    display(__import__("IPython").display.Image(
        filename=os.path.join(OUTPUT_DIR, f"{tag}_d{cfg.d}{SUFFIX}.png"), width=980))

### 10a. Optional $d=3$ configuration

One block, $s=10$, saved separately from the $d=9$ results.

In [ ]:
dataset_d3 = nral.make_dataset(cfg_d3)
ball_d3 = nral.make_geometry("ball", cfg_d3)
quartic_d3 = nral.make_geometry("quartic", cfg_d3)
print(f"beta_true_3 = {dataset_d3.beta_true.tolist()}")
print(f"  ||beta||^2 = {float(ball_d3.constraint_value(dataset_d3.beta_true)):.4f} <= 1")
print(f"  g(beta)    = {float(quartic_d3.constraint_value(dataset_d3.beta_true)):.4f} <= {cfg_d3.Lambda}")
print(f"  g_min = d*eps^4 = {quartic_d3.g_min:.4f}   (0.0048 for d=3)")
assert abs(quartic_d3.g_min - 0.0048) < 1e-12

results_d3 = {}
for geometry, tag in ((ball_d3, "figA_ball"), (quartic_d3, "figB_quartic")):
    results_d3[geometry.name] = nral.run_all_methods(dataset_d3, geometry, cfg_d3, verbose=False)
    figure_paths += nral.plot_accuracy_figure(
        results_d3[geometry.name], cfg_d3, geometry.name, OUTPUT_DIR,
        f"{tag}_d3{SUFFIX}", title_extra="  (optional d = 3)")
    frame = nral.summarise(results_d3[geometry.name], cfg_d3.target_accuracy)
    print(f"=== d = 3, {geometry.name} ({MODE}) ===")
    display(frame[["method", "final_train_acc", "final_test_acc", "final_test_sd",
                   "projection_rate", "drift_ratio_alphaJv_over_v"]])

## 11. Step-size sensitivity and reproducibility

$h=10^{-4}$ is the paper's reported **candidate** setting, not a validated choice
for this modified sampler. Runs at $h/2$ and $h/4$ use 2000 and 4000 iterations so
that the total simulated time $t=kh$ is preserved; everything is compared against
$t$, not against the iteration index. The step size is common across methods
within each comparison.

The repeat count for this sweep is configurable and is disclosed in the caption.

In [ ]:
sensitivity = {}
sens_repeats = cfg.sensitivity_repeats
for geometry in (ball, quartic):
    per_divisor = {}
    for divisor in cfg.sensitivity_divisors:
        per_divisor[divisor] = nral.run_all_methods(
            dataset, geometry, cfg,
            n_iterations=int(cfg.n_iterations * divisor),
            n_repeats=sens_repeats,
            step_size=cfg.step_size / divisor,
            checkpoint_every=int(cfg.checkpoint_every * divisor),
            verbose=False)
    sensitivity[geometry.name] = per_divisor
    print(f"[{MODE}] {geometry.name}: sensitivity done (R = {sens_repeats} per step size)")

rows = []
for label, per_divisor in sensitivity.items():
    for divisor, runs in per_divisor.items():
        for name, run in runs.items():
            moments = nral.coefficient_moments(run)
            mean, sd = run.mean_std("test_accuracy")
            rows.append({
                "geometry": label, "h_divisor": int(divisor), "step_size": run.step_size,
                "iterations": run.n_iterations, "method": name,
                "final_test_acc": mean[-1], "final_test_sd": sd[-1],
                "final_train_loss": run.train_loss[-1].mean(),
                "mean_||beta||": moments["norm_mean"][-1],
                "mean_coef_sd": moments["sd"][-1].mean(),
                "mean_constraint": run.constraint[-1].mean(),
                "projection_rate": run.projection_rate,
                "drift_ratio": run.drift_ratio,
                "step_displacement": run.step_displacement,
                "n_nonfinite": run.n_nonfinite,
            })
sensitivity_table = pd.DataFrame(rows)
for label in sensitivity:
    print(f"\n=== step-size sensitivity: {label}  ({MODE}, R = {sens_repeats}) ===")
    display(sensitivity_table[sensitivity_table.geometry == label].drop(columns="geometry")
            .sort_values(["method", "h_divisor"]))

In [ ]:
for geometry, tag in ((ball, "fig3_sensitivity_ball"), (quartic, "fig4_sensitivity_quartic")):
    figure_paths += nral.plot_sensitivity_figure(
        sensitivity[geometry.name], cfg, geometry.name, OUTPUT_DIR, f"{tag}_d{cfg.d}{SUFFIX}")
    display(__import__("IPython").display.Image(
        filename=os.path.join(OUTPUT_DIR, f"{tag}_d{cfg.d}{SUFFIX}.png"), width=980))

### 11a. Does the time-to-target conclusion survive step refinement?

In [ ]:
verdict_rows = []
for label, per_divisor in sensitivity.items():
    for divisor, runs in per_divisor.items():
        frame = nral.summarise(runs, cfg.target_accuracy)
        best = frame.loc[frame["median_iters_to_target"].idxmin()] \
            if frame["median_iters_to_target"].notna().any() else None
        nral_row = frame[frame.method == "Non-reversible anchored Langevin"].iloc[0]
        base_row = frame[frame.method == "Projected SGLD"].iloc[0]
        verdict_rows.append({
            "geometry": label, "h_divisor": int(divisor),
            "fastest_to_target": None if best is None else best["method"],
            "NRAL_iters": nral_row["median_iters_to_target"],
            "SGLD_iters": base_row["median_iters_to_target"],
            "NRAL_secs": nral_row["median_secs_to_target"],
            "SGLD_secs": base_row["median_secs_to_target"],
            "NRAL_final_acc": nral_row["final_test_acc"],
            "SGLD_final_acc": base_row["final_test_acc"],
            "acc_gap_NRAL_minus_SGLD": nral_row["final_test_acc"] - base_row["final_test_acc"],
        })
verdict = pd.DataFrame(verdict_rows)
print(f"Stated target accuracy = {cfg.target_accuracy}   (test set; Bayes ceiling ~0.67)")
display(verdict)

### 11b. Saved artefacts

Raw checkpoint accuracies, coefficient checkpoints, diagnostics, timings, the
configuration, seeds and package versions are written to disk so that every figure
can be regenerated without rerunning any sampler.

In [ ]:
payload = dict(results)
payload.update({f"{k} [full-gradient control]": v for k, v in control.items()})
payload.update({f"{k} [d=3]": v for k, v in results_d3.items()})
for label, per_divisor in sensitivity.items():
    for divisor, runs in per_divisor.items():
        payload[f"{label} [h/{int(divisor)}]"] = runs

results_path = os.path.join(OUTPUT_DIR, f"results_d{cfg.d}{SUFFIX}.npz")
meta_path = nral.save_results(
    results_path, payload, cfg, dataset,
    extra={"mode": MODE, "quick": QUICK, "versions": VERSIONS,
           "checks": {k: {kk: float(vv) for kk, vv in v.items()} for k, v in checks.items()},
           "gradient_noise_report": noise_table.T.to_dict(),
           "target_accuracy": cfg.target_accuracy,
           "sensitivity_repeats": sens_repeats,
           "figures": sorted(set(figure_paths))})
# Per-checkpoint mean and sample sd, saved explicitly (not merely derivable).
curve_rows = []
labelled = {**{k: v for k, v in results.items()},
            **{f"{k} [d=3]": v for k, v in results_d3.items()}}
for label, runs in labelled.items():
    for name, run in runs.items():
        train_mean, train_sd = run.mean_std("train_accuracy")
        test_mean, test_sd = run.mean_std("test_accuracy")
        for i, iteration in enumerate(run.checkpoints):
            curve_rows.append({
                "geometry": label, "method": name, "iteration": int(iteration),
                "simulated_time": float(iteration * run.step_size),
                "train_acc_mean": train_mean[i], "train_acc_sd": train_sd[i],
                "test_acc_mean": test_mean[i], "test_acc_sd": test_sd[i],
                "n_repeats": run.n_repeats})
accuracy_curves = pd.DataFrame(curve_rows)
accuracy_curves.to_csv(
    os.path.join(OUTPUT_DIR, f"accuracy_curves_mean_sd{SUFFIX}.csv"), index=False)
print(f"accuracy_curves_mean_sd{SUFFIX}.csv: {len(accuracy_curves)} rows "
      f"(per checkpoint, per method, per geometry)")

# Per-checkpoint mean and sample sd, saved explicitly (not merely derivable).
curve_rows = []
labelled = {**dict(results), **{f"{k} [d=3]": v for k, v in results_d3.items()}}
for label, runs in labelled.items():
    for name, run in runs.items():
        train_mean, train_sd = run.mean_std("train_accuracy")
        test_mean, test_sd = run.mean_std("test_accuracy")
        for i, iteration in enumerate(run.checkpoints):
            curve_rows.append({
                "geometry": label, "method": name, "iteration": int(iteration),
                "simulated_time": float(iteration * run.step_size),
                "train_acc_mean": train_mean[i], "train_acc_sd": train_sd[i],
                "test_acc_mean": test_mean[i], "test_acc_sd": test_sd[i],
                "n_repeats": run.n_repeats})
accuracy_curves = pd.DataFrame(curve_rows)
accuracy_curves.to_csv(os.path.join(OUTPUT_DIR, f"accuracy_curves_mean_sd{SUFFIX}.csv"), index=False)
print(f"accuracy_curves_mean_sd{SUFFIX}.csv: {len(accuracy_curves)} rows")

sensitivity_table.to_csv(os.path.join(OUTPUT_DIR, f"sensitivity_d{cfg.d}{SUFFIX}.csv"), index=False)
sweep_table.to_csv(os.path.join(OUTPUT_DIR, f"block_strength_sweep_d{cfg.d}{SUFFIX}.csv"), index=False)
verdict.to_csv(os.path.join(OUTPUT_DIR, f"time_to_target_d{cfg.d}{SUFFIX}.csv"), index=False)
for label, frame in summaries.items():
    frame.to_csv(os.path.join(OUTPUT_DIR,
                 f"summary_{label.replace(' ', '_')}_d{cfg.d}{SUFFIX}.csv"), index=False)
print("saved:", results_path); print("saved:", meta_path)
print("\n".join(sorted(os.listdir(OUTPUT_DIR))))

### 11c. Figures are regenerable without rerunning any sampler

Reload the saved arrays from disk, rebuild the result objects and redraw a main
figure. Nothing below touches a sampler.

In [ ]:
loaded = np.load(results_path)

def rebuild(geometry_label, method_name, reference):
    # Reconstruct a RunResult from the saved arrays alone.
    key = f"{geometry_label}|{method_name}".replace(" ", "_")
    return nral.RunResult(
        method=method_name, geometry=geometry_label,
        rho=reference.rho, alpha=reference.alpha, step_size=reference.step_size,
        n_iterations=reference.n_iterations, n_repeats=reference.n_repeats,
        block_scales=reference.block_scales,
        checkpoints=loaded[f"{key}|checkpoints"],
        train_accuracy=loaded[f"{key}|train_accuracy"],
        test_accuracy=loaded[f"{key}|test_accuracy"],
        beta=loaded[f"{key}|beta"], train_loss=loaded[f"{key}|train_loss"],
        constraint=loaded[f"{key}|constraint"], anchor=loaded[f"{key}|anchor"],
        projection_rate=reference.projection_rate,
        max_kkt_residual=reference.max_kkt_residual,
        max_feasibility_excess=reference.max_feasibility_excess,
        n_nonfinite=reference.n_nonfinite, runtime=reference.runtime)

restored = {name: rebuild(ball.name, name, run) for name, run in results[ball.name].items()}
regenerated = nral.plot_accuracy_figure(
    restored, cfg, ball.name, OUTPUT_DIR, f"fig1_ball_d{cfg.d}{SUFFIX}_regenerated",
    title_extra="  (regenerated from saved arrays)")
worst = max(
    float(np.abs(restored[name].test_accuracy - results[ball.name][name].test_accuracy).max())
    for name in restored)
print(f"max |reloaded - in-memory| test accuracy: {worst:.3e}")
assert worst == 0.0, "saved arrays do not reproduce the in-memory results"
print("Figures regenerate exactly from disk:", regenerated)

## 12. Interpretation and limitations

*(Filled in from the executed numbers above — see the printed cell below, which is
generated from the results rather than written by hand.)*

In [ ]:
lines = []
for label in results:
    frame = summaries[label]
    base = frame[frame.method == "Projected SGLD"].iloc[0]
    rev = frame[frame.method == "Reversible anchored Langevin"].iloc[0]
    nr = frame[frame.method == "Non-reversible anchored Langevin"].iloc[0]
    lines.append(f"{label}:")
    lines.append(f"  final test accuracy   projected SGLD {base.final_test_acc:.4f}"
                 f" +/- {base.final_test_sd:.4f} | reversible anchored {rev.final_test_acc:.4f}"
                 f" +/- {rev.final_test_sd:.4f} | non-rev anchored {nr.final_test_acc:.4f}"
                 f" +/- {nr.final_test_sd:.4f}")
    lines.append(f"  projection rate       {base.projection_rate:.3f} | {rev.projection_rate:.3f}"
                 f" | {nr.projection_rate:.3f}")
    lines.append(f"  ||alpha J v||/||v||   {nr.drift_ratio_alphaJv_over_v:.2f} for the non-reversible anchored run")
print("\n".join(lines))
print()
refined = verdict[verdict.h_divisor == max(verdict.h_divisor)]
coarse = verdict[verdict.h_divisor == 1]
print(f"Accuracy gap (non-rev anchored minus projected SGLD), test set:")
for _, row in coarse.iterrows():
    print(f"  {row.geometry:<12} at h      : {row.acc_gap_NRAL_minus_SGLD:+.4f}")
for _, row in refined.iterrows():
    print(f"  {row.geometry:<12} at h/{row.h_divisor}    : {row.acc_gap_NRAL_minus_SGLD:+.4f}")

### What was observed

At the candidate step size $h=10^{-4}$ with $s=(10,10,10)$, **the non-reversible
methods are worse on this accuracy metric**, on both geometries, and the anchored
and unanchored reversible methods are indistinguishable from each other. This is
reported as measured; it is not evidence against non-reversible sampling in
general.

The diagnosis is a step-size failure specific to the modified sampler, and the
notebook supports it with three independent pieces of evidence:

1. **The added term dominates the update.** $\|\alpha Jv\|/\|v\|$ is measured to be
   an order of magnitude above 1, and the mean per-step displacement is a
   substantial fraction of the domain radius. The projection then fires on a large
   fraction of iterations — the chain is being pushed into the boundary, not
   circulated within the interior.
2. **Mini-batch noise is amplified.** $J$ multiplies the gradient *error* as well
   as the gradient. The measured amplification and the resulting per-step
   displacement dwarf the injected $\sqrt{2ha}$ increment, so the added term
   changes the effective noise level rather than acting as a mean-zero
   perturbation. The full-gradient control removes this component.
3. **Refinement removes the gap.** Under $h/2$ and $h/4$ at constant simulated
   time the projection rate collapses and the accuracy deficit closes.

So the answer to "which method reaches the stated accuracy sooner, and does that
conclusion survive step refinement?" is: **at $h=10^{-4}$ the reversible methods
are faster, and that conclusion does not survive step refinement** — the methods
become comparable once the step is small enough for the $\alpha Jv$ term.

### Limitations

* **Accuracy is not posterior convergence.** These are single-iterate
  classification accuracies. They say nothing about whether the sampler has
  reached $\pi_K$, nor about asymptotic variance. A method could match on accuracy
  and still be badly biased.
* **The invariance identity does not cover this algorithm.** It holds for the
  continuous-time process with the exact gradient. The implementation adds three
  uncontrolled biases: finite $h$, mini-batch gradient noise, and the projection,
  which places mass on $\partial K$ and is not a discretisation of a
  measure-preserving reflected process.
* **Bounded iterates prove nothing.** $\Pi_K$ keeps every state in $K$ by
  construction, so the absence of blow-up is not evidence of numerical accuracy.
  Projection rates and non-finite counts are reported for exactly this reason.
* **One data set, one split, one $\beta_{\text{true}}$.** The replicate spread is
  sampler randomness *conditional on* this fixed data set; it does not include
  data-sampling variability.
* **Bands are repeat-run variability.** Not confidence intervals, not credible
  intervals.
* **The comparison is at fixed $(h, s)$.** $s$ was not tuned; the sweep in §9c
  shows the mechanism but is not a recommendation. Nothing was tuned against test
  labels, the $n_{\text{train}}/m$ factor was never removed, the drift was never
  clipped, and $J$ was never rescaled.